# DS300 FacetRec — Tải Base Models Part 1 (Qwen3-4B + Qwen3-8B + BERT)

Tải sẵn **`Qwen3-4B`** (~7.6 GB) + **`Qwen3-8B`** (~15.3 GB) + **`bert-base-uncased`** (~0.44 GB) ≈ **23.3 GB** và tự động đẩy thành **Private Dataset** dùng cho `kaggle_run_demo.ipynb`.

### Hướng dẫn thiết lập 1 lần trước khi chạy (không cần bật GPU!)
1. **Notebook options → Accelerator:** Chọn **`None`** (chỉ tải & đẩy Dataset, không tốn quota GPU).
2. **Notebook options → Internet:** Bật **`ON`**.
3. **Add-ons → Secrets:** Thêm 2 Secret để tự động tạo Private Dataset:
   - **`KAGGLE_USERNAME`**: tên tài khoản Kaggle của bạn (xem trong *Settings → Account*, vd: `amalkin39`).
   - **`KAGGLE_KEY`**: mở *Kaggle → Settings → API → Create New Token* để tải file `kaggle.json`, mở file đó copy giá trị trường `"key"`.
4. *(Không cần `HF_TOKEN` vì Qwen3 & BERT là model mở.)*
5. Bấm **Run All** — notebook sẽ tải thẳng vào `/tmp` (không bị giới hạn 19.5 GB của `/kaggle/working`) và tự đẩy thành **Private Dataset** trên tài khoản Kaggle của bạn!

In [ ]:
# ==============================================================================
# CELL 1: Cấu hình & Đọc Secrets (KAGGLE_USERNAME, KAGGLE_KEY)
# ==============================================================================
import json
import os
import shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

DATASET_SLUG = "ds300-base-models-part1-qwen"
DATASET_TITLE = "DS300 Base Models Part 1 Qwen"
EXPORT_ROOT = Path("/tmp/ds300_part1")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

MODELS = [
    ("Qwen3-4B", "Qwen/Qwen3-4B"),
    ("Qwen3-8B", "Qwen/Qwen3-8B"),
    ("bert-base-uncased", "google-bert/bert-base-uncased"),
]

secrets = UserSecretsClient()
KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME").strip()
KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY").strip()
assert KAGGLE_USERNAME and KAGGLE_KEY, "Thiếu KAGGLE_USERNAME hoặc KAGGLE_KEY trong Kaggle Secrets!"

# Ghi ~/.kaggle/kaggle.json cho Kaggle CLI
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
os.chmod(kaggle_json, 0o600)
print(f"✅ Đã cấu hình Kaggle API cho tài khoản: {KAGGLE_USERNAME}")

HF_TOKEN = ""
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN").strip()
    print("✅ Đã đọc HF_TOKEN từ Kaggle Secrets.")
except Exception:
    print('ℹ️ Không có HF_TOKEN (không cần cho Qwen3 & BERT).')

free_tmp = shutil.disk_usage("/tmp").free / (1024**3)
free_work = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"💾 Ổ trống: /tmp = {free_tmp:.1f} GB | /kaggle/working = {free_work:.1f} GB")

In [ ]:
# ==============================================================================
# CELL 2: Tải Snapshot phẳng vào /tmp (tốc độ cao với hf_transfer, bỏ file thừa)
# ==============================================================================
import os
import shutil
import subprocess
from pathlib import Path

subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub>=0.25.0", "hf_transfer", "kaggle"], check=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

IGNORE_PATTERNS = [
    "*.msgpack",
    "*.h5",
    "*.ot",
    "*.onnx",
    "*.tflite",
    "tf_model*",
    "flax_model*",
    "rust_model*",
    "original/*",
    "pytorch_model*.bin",
]

for folder_name, repo_id in MODELS:
    target_dir = EXPORT_ROOT / folder_name
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n📥 Đang tải {repo_id} -> {target_dir} ...")

    ignores = [
        p for p in IGNORE_PATTERNS
        if not (folder_name == "bert-base-uncased" and p == "pytorch_model*.bin")
    ]

    snapshot_download(
        repo_id=repo_id,
        local_dir=str(target_dir),
        token=HF_TOKEN or None,
        ignore_patterns=ignores,
    )

    dot_cache = target_dir / ".cache"
    if dot_cache.exists():
        shutil.rmtree(dot_cache, ignore_errors=True)

    size_gb = sum(f.stat().st_size for f in target_dir.rglob("*") if f.is_file()) / (1024**3)
    print(f"✅ Xong {folder_name} ({size_gb:.2f} GB)!")

In [ ]:
# ==============================================================================
# CELL 3: Kiểm tra đủ file & Tự động đẩy thành Private Dataset trên Kaggle
# ==============================================================================
import json
import os
import shutil
import subprocess
from pathlib import Path

total_gb = 0.0
for folder_name, _ in MODELS:
    d = EXPORT_ROOT / folder_name
    has_cfg = (d / "config.json").exists()
    weights = list(d.glob("*.safetensors")) + list(d.glob("*.bin"))
    size_gb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / (1024**3)
    total_gb += size_gb
    assert has_cfg and len(weights) > 0, f"❌ Thiếu config.json hoặc weights trong {d}"
    print(f" ✅ {folder_name:24s} | {size_gb:5.2f} GB | {len(weights)} weight file(s)")

print(f"\n📊 Tổng dung lượng chuẩn bị upload: {total_gb:.2f} GB")

# Ghi metadata Private Dataset
meta = {
    "title": DATASET_TITLE,
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "other"}],
}
(EXPORT_ROOT / "dataset-metadata.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Đặt TMPDIR trên /tmp (trống hàng trăm GB) cho bước đóng gói tar
tmp_upload = Path("/tmp/ds300_upload_tmp")
tmp_upload.mkdir(parents=True, exist_ok=True)
env_up = os.environ.copy()
env_up["TMPDIR"] = str(tmp_upload)

# Kiểm tra xem Dataset đã tồn tại chưa -> tạo mới hoặc cập nhật phiên bản mới
check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    env=env_up,
    capture_output=True,
    text=True,
)

if check.returncode == 0:
    cmd = [
        "kaggle", "datasets", "version",
        "-p", str(EXPORT_ROOT),
        "-m", "Update flat model snapshots",
        "--dir-mode", "tar",
    ]
    print("🔄 Dataset đã tồn tại -> đang cập nhật phiên bản mới...")
else:
    cmd = [
        "kaggle", "datasets", "create",
        "-p", str(EXPORT_ROOT),
        "--dir-mode", "tar",
    ]
    print("🚀 Đang tạo Private Dataset mới trên Kaggle (mất vài phút để upload)...")

try:
    subprocess.run(cmd, env=env_up, check=True)
    print("\n" + "=" * 72)
    print(f"🎉 HOÀN TẤT! Private Dataset đã sẵn sàng:")
    print(f"👉 https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")
    print("💡 Mở kaggle_run_demo.ipynb -> Add Input -> Datasets -> chọn Dataset này là xong!")
    print("=" * 72)
finally:
    shutil.rmtree(tmp_upload, ignore_errors=True)